# W07 - Multiple Regression: The Least Squares Method

## 1. Introduction: Extending the Least Squares Principle

When moving from simple linear regression to multiple linear regression, the central optimization principle remains unchanged: Find the model that minimizes prediction error.

The method used is still Ordinary Least Squares (OLS). The only difference is geometric complexity.

Simple Linear Regression fits a line: y_hat = b0 + b1 * x
Multiple Linear Regression fits a hyperplane: y_hat = b0 + b1 * x1 + b2 * x2 + ... + bk * xk

In higher dimensions, OLS attempts to place this hyperplane so that it lies as close as possible to the observed data points.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import statsmodels.api as sm

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set random seed for reproducibility
np.random.seed(42)

print("Environment setup complete. Libraries imported successfully.")

## 2. Data Creation: Synthetic Performance Dataset

To understand the math behind multiple regression, we need data. We will create a synthetic dataset representing student exam scores based on three predictors:
1. Study Hours per week
2. Attendance Percentage
3. Prior GPA

We will intentionally build the Target (Exam Score) as a linear combination of these factors, plus some random noise (epsilon).

In [ ]:
# Number of observations
n_samples = 250

# Generate Predictors
study_hours = np.random.normal(15, 5, n_samples)
attendance_pct = np.clip(np.random.normal(85, 10, n_samples), 0, 100)
prior_gpa = np.random.uniform(2.0, 4.0, n_samples)

# True Population Parameters (The hidden truth we want our model to find)
true_b0 = 10.0      # Base score
true_b1 = 1.5       # Score increase per study hour
true_b2 = 0.3       # Score increase per attendance %
true_b3 = 8.0       # Score increase per GPA point

# Generate Random Error (Residual noise)
epsilon = np.random.normal(0, 4.5, n_samples)

# Generate Target (y)
exam_score = true_b0 + (true_b1 * study_hours) + (true_b2 * attendance_pct) + (true_b3 * prior_gpa) + epsilon

# Assemble into a DataFrame
df = pd.DataFrame({
    'Study_Hours': study_hours,
    'Attendance_Pct': attendance_pct,
    'Prior_GPA': prior_gpa,
    'Exam_Score': exam_score
})

print("Synthetic dataset generated successfully!")
print(df.head())

## 3. Residual Errors and SSE (Sum of Squared Errors)

Closeness in regression is measured using Residual Errors.
For any observation i:
e_i = y_i - y_hat_i

Where:
- y_i represents the Actual value
- y_hat_i represents the Predicted value
- e_i represents the Residual

OLS minimizes the Sum of Squared Errors (SSE):
SSE = sum((y_i - y_hat_i)^2)

Let's see what happens if we guess coefficients poorly versus accurately.

In [ ]:
def calculate_sse(b0, b1, b2, b3, data):
    # Generate predictions based on guessed coefficients
    y_hat = b0 + (b1 * data['Study_Hours']) + (b2 * data['Attendance_Pct']) + (b3 * data['Prior_GPA'])
    # Calculate residuals
    residuals = data['Exam_Score'] - y_hat
    # Return SSE
    return np.sum(residuals ** 2)

# 1. A terrible guess
bad_sse = calculate_sse(0, 0, 0, 0, df)

# 2. A very good guess (close to true parameters)
good_sse = calculate_sse(10, 1.5, 0.3, 8.0, df)

print(f"SSE for a terrible guess (all zeros): {bad_sse:,.2f}")
print(f"SSE for a great guess (true params):  {good_sse:,.2f}")
print("\nOur goal with OLS is to find the mathematical minimum of this SSE.")

## 4. Why Squaring Matters

Squaring residuals serves several crucial purposes:
1. Prevents Cancellation: Positive errors and negative errors would cancel each other out if we simply summed them.
2. Penalizes Large Errors: Large mistakes become disproportionately costly. An error of 2 becomes 4, but an error of 10 becomes 100.
3. Produces Smooth Optimization: Squared functions are differentiable, which makes calculus and exact mathematical solutions possible.

Let's visualize how the penalty grows.

In [ ]:
# Visualizing the squared penalty
error_values = np.linspace(-10, 10, 100)
absolute_penalty = np.abs(error_values)
squared_penalty = error_values ** 2

plt.figure(figsize=(8, 5))
plt.plot(error_values, absolute_penalty, label='Absolute Error |e|', color='blue')
plt.plot(error_values, squared_penalty, label='Squared Error (e^2)', color='red')
plt.title('Why We Use Least Squares: Penalty Growth')
plt.xlabel('Residual Error (e)')
plt.ylabel('Penalty Applied')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Notice how the red curve (Squared Error) heavily punishes outliers (e = 10) compared to the blue curve.")

## 5. Matrix Representation of Regression

Writing out partial derivatives for every variable becomes impossible by hand. Modern regression depends fundamentally on Matrix Algebra.

The regression model is represented compactly as:
y = X * b + e

Where:
- y is the Response Vector (n x 1)
- X is the Design Matrix (n x k+1) - containing all predictors and a leading column of 1s for the intercept.
- b is the Coefficient Vector (k+1 x 1)
- e is the Residual Vector (n x 1)

Let's build the Design Matrix (X) and Response Vector (y) in Python.

In [ ]:
# Extract response vector y
y_vec = df['Exam_Score'].values

# Extract predictor columns
predictors = df[['Study_Hours', 'Attendance_Pct', 'Prior_GPA']].values

# The Design Matrix X requires a column of 1s at the beginning for the intercept (b0)
ones_column = np.ones((n_samples, 1))
X_mat = np.hstack((ones_column, predictors))

print(f"Shape of Design Matrix X: {X_mat.shape}")
print(f"Shape of Response Vector y: {y_vec.shape}")
print("\nFirst 3 rows of Design Matrix X:")
print(X_mat[:3])
print("\n(Notice the leading column of 1.0s - this allows the matrix math to calculate the intercept)")

## 6. The Normal Equations: Closed-Form Solution

Using matrix calculus, we differentiate SSE with respect to the vector b and set it to zero. This produces the Matrix Normal Equations:
(X^T * X) * b = X^T * y

Solving for b, we get the famous Closed-Form OLS Solution:
b = (X^T * X)^-1 * X^T * y

This single equation finds the absolute optimal coefficients in one mathematical step!

In [ ]:
# Manual Matrix Math to solve for b

# Step 1: Calculate X^T * X
X_T_X = X_mat.T @ X_mat

# Step 2: Calculate the inverse of (X^T * X)
X_T_X_inv = np.linalg.inv(X_T_X)

# Step 3: Calculate X^T * y
X_T_y = X_mat.T @ y_vec

# Step 4: Multiply to get the coefficient vector b
b_hat = X_T_X_inv @ X_T_y

print("--- MANUALLY CALCULATED COEFFICIENTS (b_hat) ---")
print(f"Intercept (b0):     {b_hat[0]:.4f}")
print(f"Study_Hours (b1):   {b_hat[1]:.4f}")
print(f"Attendance_Pct (b2):{b_hat[2]:.4f}")
print(f"Prior_GPA (b3):     {b_hat[3]:.4f}")

## 7. Verifying the Matrix Math with standard Libraries

Does our manual linear algebra match what industry-standard libraries like `statsmodels` output? Let's check.

In [ ]:
# Using statsmodels to verify our math
model = sm.OLS(y_vec, X_mat).fit()

print("--- STATSMODELS COEFFICIENTS ---")
print(f"Intercept (b0):     {model.params[0]:.4f}")
print(f"Study_Hours (b1):   {model.params[1]:.4f}")
print(f"Attendance_Pct (b2):{model.params[2]:.4f}")
print(f"Prior_GPA (b3):     {model.params[3]:.4f}")

print("\nConclusion: Our manual matrix algebra perfectly matches the professional library!")
print("This equation is the mathematical backbone of Linear Regression.")

## 8. Geometric Interpretation: Orthogonality

OLS is fundamentally a projection problem. The fitted values (y_hat) are the orthogonal projection of y onto the column space of X.

Because it is an orthogonal projection, the residual vector (e) is completely perpendicular (orthogonal) to the predictor space.

Mathematically, this means the dot product of X transpose and e is the zero vector:
X^T * e = 0

Let's prove this deep geometric truth using our dataset.

In [ ]:
# 1. Calculate the predictions (y_hat = X * b)
y_hat = X_mat @ b_hat

# 2. Calculate the exact residuals (e = y - y_hat)
residuals_vec = y_vec - y_hat

# 3. Compute the dot product: X^T * e
orthogonality_check = X_mat.T @ residuals_vec

print("Result of X^T * e (Orthogonality Check):")
for i, val in enumerate(orthogonality_check):
    print(f"Component {i}: {val:.10f}")

print("\nAs you can see, the values are effectively zero (e.g., e-12 is 0.000000000001).")
print("This proves the residuals are orthogonal to the predictors! Geometry works.")

## 9. Understanding the components: (X^T * X)

The matrix (X^T * X) captures the variances of predictors and the covariances between them. It describes the predictor geometry.

Inverting it, (X^T * X)^-1, adjusts for predictor overlap and disentangles shared information between variables.

In [ ]:
print("The X^T * X Matrix:")
df_xtx = pd.DataFrame(X_T_X, 
                      columns=['Intercept', 'Study', 'Attend', 'GPA'],
                      index=['Intercept', 'Study', 'Attend', 'GPA'])
print(df_xtx.round(1))

print("\nNotice that the diagonal contains the sum of squares of each feature, and the off-diagonals contain the cross-products.")

## 10. The Danger of Multicollinearity

What happens if predictors are highly correlated or identical? 
The matrix (X^T * X) becomes nearly singular (non-invertible).
Its inverse becomes extremely unstable, leading to wildly fluctuating coefficients.

This is the mathematical root of Multicollinearity.

In [ ]:
# Let's intentionally break the math by adding a completely redundant feature
# Study_Minutes is just Study_Hours * 60 (perfect collinearity)
study_minutes = study_hours * 60

# Build the flawed Design Matrix
X_flawed = np.column_stack((X_mat, study_minutes))
X_flawed_T_X_flawed = X_flawed.T @ X_flawed

print("Attempting to invert (X^T * X) with perfect multicollinearity...")
try:
    inv_flawed = np.linalg.inv(X_flawed_T_X_flawed)
    print("Inversion successful.")
except np.linalg.LinAlgError as e:
    print(f"\nERROR TRIGGERED: {e}")
    print("Because columns are linearly dependent, the matrix determinant is zero, making inversion impossible!")
    print("This is why data scientists must remove perfectly correlated features.")

## 11. Computational Perspective and Limits

Direct matrix inversion is elegant, but it has limits.
The time complexity to invert a matrix is roughly O(k^3), where k is the number of features.

For datasets with thousands of features (like deep learning or genomics), direct inversion becomes too slow. Large-scale ML systems use Gradient Descent instead of direct inversion.

Let's demonstrate how inversion time scales.

In [ ]:
# Benchmarking matrix inversion scaling
feature_sizes = [10, 100, 500, 1000, 2000]
times = []

for k in feature_sizes:
    # Create a random (k x k) matrix
    random_matrix = np.random.rand(k, k)
    # Ensure it's invertible by adding to the diagonal
    random_matrix += np.eye(k) * k
    
    start_time = time.time()
    _ = np.linalg.inv(random_matrix)
    end_time = time.time()
    
    times.append(end_time - start_time)

plt.figure(figsize=(8, 4))
plt.plot(feature_sizes, times, marker='o', color='purple', linewidth=2)
plt.title('Computational Cost: Time to Invert (X^T * X)')
plt.xlabel('Number of Features (k)')
plt.ylabel('Time in Seconds')
plt.grid(alpha=0.3)
plt.show()

print("Notice the cubic curve. As features grow, direct inversion time explodes.")

## 12. Residual Analysis

Even though we found the mathematically optimal OLS plane, we must check if our model assumptions are valid by analyzing the residuals.

In [ ]:
# Visualizing Residuals
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# 1. Histogram of Residuals
sns.histplot(residuals_vec, kde=True, ax=axs[0], color='teal')
axs[0].set_title('Distribution of Residuals')
axs[0].set_xlabel('Residual Error (e)')
axs[0].grid(alpha=0.3)

# 2. Residuals vs Predicted (Homoscedasticity check)
axs[1].scatter(y_hat, residuals_vec, alpha=0.5, color='darkorange')
axs[1].axhline(0, color='red', linestyle='--')
axs[1].set_title('Residuals vs Predicted Values')
axs[1].set_xlabel('Predicted Exam Score (y_hat)')
axs[1].set_ylabel('Residual Error (e)')
axs[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Good Residuals: They are normally distributed around zero, with no clear patterns in the scatter plot.")

## 13. Practice Exercises

### Exercise 1: Manual Prediction
Using the `b_hat` coefficients calculated earlier, manually predict the Exam Score for a new student with:
- Study Hours: 20
- Attendance Pct: 95
- Prior GPA: 3.5

In [ ]:
# Exercise 1 Setup
student_features = [1, 20, 95, 3.5]  # Note the 1 for the intercept!
print("Goal: Perform a dot product to predict the score.")
# Your code here...

In [ ]:
# Exercise 1 Solution
student_vec = np.array(student_features)
predicted_score = student_vec @ b_hat
print(f"Predicted Exam Score: {predicted_score:.2f}")

### Exercise 2: Implementing SSE as a Vector Operation
Instead of writing a loop, calculate the SSE for our model using pure vector operations: e^T * e.

In [ ]:
# Exercise 2 Setup
print("Goal: Compute e^T * e")
# Your code here...

In [ ]:
# Exercise 2 Solution
# We already have residuals_vec calculated
sse_vectorized = residuals_vec.T @ residuals_vec
print(f"Vectorized SSE: {sse_vectorized:,.2f}")
print("This is the absolute mathematically lowest possible SSE for this dataset!")

## 14. Summary and Key Takeaways

- Least squares is fundamentally geometric projection + optimization.
- The OLS solution: b = (X^T * X)^-1 * X^T * y
- This finds the coefficient vector that minimizes the Sum of Squared Errors: (y - Xb)^T * (y - Xb)
- Regression minimizes squared residuals because it prevents cancellation, penalizes large mistakes, and provides a smooth optimization surface.
- Residuals are always geometrically orthogonal to the predictor space.
- If predictors are collinear, (X^T * X) cannot be inverted, crashing the algorithm.
- This matrix framework forms the mathematical backbone of modern statistical learning systems and deep learning.

## 15. Visualization Gallery
A final gallery showing how the components relate.

In [ ]:
fig = plt.figure(figsize=(15, 5))

# 1. Actual vs Predicted
ax1 = fig.add_subplot(131)
ax1.scatter(y_vec, y_hat, alpha=0.6, color='blue')
ax1.plot([y_vec.min(), y_vec.max()], [y_vec.min(), y_vec.max()], 'r--', lw=2)
ax1.set_title('Actual vs Predicted')
ax1.set_xlabel('Actual Exam Score')
ax1.set_ylabel('Predicted Exam Score')
ax1.grid(alpha=0.3)

# 2. Predictor 1 vs Target
ax2 = fig.add_subplot(132)
sns.regplot(x=df['Study_Hours'], y=df['Exam_Score'], ax=ax2, 
            scatter_kws={'alpha':0.5, 'color':'green'}, 
            line_kws={'color':'red'})
ax2.set_title('Marginal Effect: Study Hours')
ax2.grid(alpha=0.3)

# 3. Predictor 3 vs Target
ax3 = fig.add_subplot(133)
sns.regplot(x=df['Prior_GPA'], y=df['Exam_Score'], ax=ax3, 
            scatter_kws={'alpha':0.5, 'color':'purple'}, 
            line_kws={'color':'red'})
ax3.set_title('Marginal Effect: Prior GPA')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Module Complete: W07 - Multiple Regression: Least Squares Method Successfully Rendered.")